In [2]:
import httpx
import time
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0617 21:58:12.068000 6052 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
theme_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    multi_label = True
)

def theme_result(text, theme_labels):
    return theme_classifier(
        text,
        candidate_labels=theme_labels,
        hypothesis_template="This post discusses {}.",
        multi_label=True
    )

model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [4]:
file_path_1 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/1st_4_subreddits_description.json")
file_path_2 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/1st_4_subreddits_title.json")
file_path_3 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/2nd_4_subreddits_description.json")
file_path_4 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/2nd_4_subreddits_title.json")
file_path_5 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/3rd_4_subreddits_description.json")
file_path_6 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/3rd_4_subreddits_title.json")
file_path_7 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/4th_4_subreddits_description.json")
file_path_8 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/4th_4_subreddits_title.json")
file_path_9 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/5th_3_subreddits_description.json")
file_path_10 = Path("/Users/mnatali/Projects/sentiment_analysis/reddit_across_subreddits_analysis/brightdata_subreddit_exports/5th_3_subreddits_title.json")


with file_path_1.open("r", encoding="utf-8") as f:
    posts_1 = json.load(f)
with file_path_2.open("r", encoding="utf-8") as f:
    posts_2 = json.load(f)
with file_path_3.open("r", encoding="utf-8") as f:
    posts_3 = json.load(f)
with file_path_4.open("r", encoding="utf-8") as f:
    posts_4 = json.load(f)
with file_path_5.open("r", encoding="utf-8") as f:  
    posts_5 = json.load(f)
with file_path_6.open("r", encoding="utf-8") as f:
    posts_6 = json.load(f)
with file_path_7.open("r", encoding="utf-8") as f:
    posts_7 = json.load(f)
with file_path_8.open("r", encoding="utf-8") as f:
    posts_8 = json.load(f)
with file_path_9.open("r", encoding="utf-8") as f:
    posts_9 = json.load(f)
with file_path_10.open("r", encoding="utf-8") as f:
    posts_10 = json.load(f)


post_lists = [posts_1, posts_2, posts_3, posts_4, posts_5, posts_6, posts_7, posts_8, posts_9, posts_10]
subreddit_posts = []
seen_post_ids = set()

for post_list in post_lists:
    for post in post_list:
        post_id = post.get("post_id")
        if post_id in seen_post_ids:
            continue
        seen_post_ids.add(post_id)
        subreddit_posts.append(post)

In [ ]:
all_post_ids = []

env_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

infr_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

not_useful_post_ids = []

themes = ["visual impact of datacenters", "infrastructure and utilities", "housing costs and property values", "economy and jobs", "quality of life, noise, and light pollution", "environmental impact", "government decisions and policies"]

# NEED TO INCLUDE THIS FILTERING
# dont_include = ["server", "rack", "switch", "firewall", "BGP", "latency"]

matched_posts = 0
more_than_one_theme_posts = 0

a = 0


for post in subreddit_posts:
    a += 1

    post_id = post["post_id"]
    all_post_ids.append(post_id)
    text = post["title"] + post["description"] if post["description"] else post["title"]

    final_labels = []

    theme_scores = theme_result(
        text,
        themes
    )
    
    for i in range(len(theme_scores['labels'])):
        if theme_scores['scores'][i] > 0.5:
            final_labels.append(theme_scores['labels'][i])
    
    if len(final_labels) > 0:
        matched_posts += 1
    else:
        not_useful_post_ids.append(post_id)
    
    if len(final_labels) > 1:
        more_than_one_theme_posts += 1
    

    for label in final_labels:
        total_sentiment = aspect_sentiment(text, label)
        post_sentiment = max(total_sentiment, key=total_sentiment.get)
        post_degree = max(total_sentiment.values())

        if label == "visual impact of datacenters":
            aesth_post_ids.append(post_id)
            aesth_post_sentiments.append(post_sentiment)
            aesth_post_sentiment_degrees.append(post_degree)
        if label == "infrastructure and utilities":
            infr_post_ids.append(post_id)
            infr_post_sentiments.append(post_sentiment)
            infr_post_sentiment_degrees.append(post_degree)
        if label == "housing costs and property values":
            housing_post_ids.append(post_id)
            housing_post_sentiments.append(post_sentiment)
            housing_post_sentiment_degrees.append(post_degree)
        if label == "economy and jobs":
            econ_post_ids.append(post_id)
            econ_post_sentiments.append(post_sentiment)
            econ_post_sentiment_degrees.append(post_degree)
        if label == "quality of life, noise, and light pollution":
            life_qual_post_ids.append(post_id)
            life_qual_post_sentiments.append(post_sentiment)
            life_qual_post_sentiment_degrees.append(post_degree)
        if label == "environmental impact":
            env_post_ids.append(post_id)
            env_post_sentiments.append(post_sentiment)
            env_post_sentiment_degrees.append(post_degree)
        if label == "government decisions and policies":
            gov_post_ids.append(post_id)
            gov_post_sentiments.append(post_sentiment)
            gov_post_sentiment_degrees.append(post_degree)

        print("Posts scanned:", a, end="\r")

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

Total posts scanned: 613
Total posts with a theme: 517
Found environmental posts: 61
Found infrastructure posts: 275
Found housing posts: 4
Found economic posts: 66
Found life quality posts: 17
Found aesthetic posts: 7
Found government posts: 266
['t3_1t3j766', 't3_1twujxp', 't3_1u2p4n7', 't3_1tjgq2d', 't3_1suhctr', 't3_1al34nd', 't3_1u16nw2', 't3_1ts1hzl', 't3_1rvvnjt', 't3_1m3cwrs', 't3_1sb2eph', 't3_1u1mj6z', 't3_1u1mzjh', 't3_13qwzhr', 't3_1thvjum', 't3_1tr4lm2', 't3_1s1usan', 't3_1sure8z', 't3_1sk5txr', 't3_1tfrvhu', 't3_1ta1rr2', 't3_1rwa7ut', 't3_1u29mgs', 't3_1sc0vni', 't3_1tjj0pr', 't3_1u1a3xn', 't3_1pezsl7', 't3_1tw2cxg', 't3_1s4m6e1', 't3_1n2anbr', 't3_1twpj0m', 't3_1thg3ys', 't3_1o8mjr1', 't3_1tu1z99', 't3_1poep1j', 't3_1ra2oyt', 't3_1q7eftn', 't3_1o1qs2n', 't3_1sxga40', 't3_1sjekqf', 't3_1smb8p7', 't3_1tuqwj5', 't3_1sgw9i8', 't3_1oujedp', 't3_1u07hcj', 't3_1n9s1bw', 't3_1t0hgkg', 't3_1tt3w7o', 't3_1tri2fj', 't3_1toagi6', 't3_1tpub0s', 't3_1mnfq7u', 't3_1qvxj14', 't3_1tinoy

In [27]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_num_comments = []
    post_upvotes = []
    post_subreddits = []

    for post in subreddit_posts:
        if post["post_id"] in theme:
            post_ids.append(post["post_id"])
            text = post["title"] + " " + post["description"] if post["description"] else post["title"]
            post_texts.append(text)
            post_dates.append(post["date_posted"])
            post_num_comments.append(post["num_comments"])
            post_upvotes.append(post["num_upvotes"])
            post_subreddits.append(post["community_name"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["number of comments"] = post_num_comments
    df1["upvotes"] = post_upvotes
    df1["subreddit"] = post_subreddits

    
    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/bd/yl2vg9fs5rbcv0trjf5f9x2m0000gp/T/ipykernel_6052/1802905402.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/bd/yl2vg9fs5rbcv0trjf5f9x2m0000gp/T/ipykernel_6052/1802905402.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/bd/yl2vg9fs5rbcv0trjf5f9x2m0000gp/T/ipykernel_6052/1802905402.py:67: FutureWarning: The behavior of DataFrame concatenation with 

In [35]:
len(posts)

517

In [36]:
grouping_cols = ["ids", "text", "date"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree"]

def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False).agg(agg)
print(len(posts))
posts.head()

517


,ids,text,date,environment,infrastructure,housing,economy,life quality,aesthetics,government,...,housing sentiment,housing sentiment degree,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree
0,t3_185gmqm,Virginia's 'Data Center Alley' residents say a...,2023-11-27T23:02:54.851Z,False,False,False,False,True,False,False,...,NaN,NaN,NaN,NaN,Negative,0.96763,NaN,NaN,NaN,NaN
1,t3_1942s4p,Top 50 Data Center Markets by Power Consumption,2024-01-11T14:19:37.275Z,False,True,False,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,t3_1dfawda,"Unfortunately I don’t live there anymore, but ...",2024-06-13T21:57:35.912Z,False,False,True,False,False,False,False,...,Positive,0.539481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,t3_1dui97s,"I like JB Pritzker, but I hate billionaires A...",2024-07-03T16:13:02.617Z,False,False,False,True,False,False,True,...,NaN,NaN,Positive,0.726162,NaN,NaN,NaN,NaN,Positive,0.717143
4,t3_1febz9m,Fairfax County board approves new data center ...,2024-09-11T14:48:08.328Z,False,True,False,False,False,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Neutral,0.938438


In [30]:
print(posts.columns)

Index(['ids', 'text', 'date', 'environment', 'infrastructure', 'housing',
       'economy', 'life quality', 'aesthetics', 'government',
       'environment sentiment', 'environment sentiment degree',
       'infrastructure sentiment', 'infrastructure sentiment degree',
       'housing sentiment', 'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree'],
      dtype='object')


In [50]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(dataset, theme):
    theme_posts = dataset.loc[dataset[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return (pos-neg)/total
    else:
        return None


print("Number of environmental posts:", len(env_post_ids))
print("Average sentiment of environmental posts:", avg_sentiment_calculation(posts, "environment"))

print("Number of infrastructure posts:", len(infr_post_ids))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation(posts, "infrastructure"))

print("Number of housing posts:", len(housing_post_ids))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation(posts, "housing"))

print("Number of economic posts:", len(econ_post_ids))
print("Average sentiment of economic posts:", avg_sentiment_calculation(posts, "economy"))

print("Number of life quality posts:", len(life_qual_post_ids))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation(posts, "life quality"))

print("Number of aesthetic posts:", len(aesth_post_ids))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation(posts, "aesthetics"))

print("Number of government posts:", len(gov_post_ids))
print("Average sentiment of governmental posts:", avg_sentiment_calculation(posts, "government"))

Number of environmental posts: 61
Average sentiment of environmental posts: -0.8102381914427238
Number of infrastructure posts: 275
Average sentiment of infrastructural posts: -0.22785041615688406
Number of housing posts: 4
Average sentiment of housing-related posts: 0.23022690224255324
Number of economic posts: 66
Average sentiment of economic posts: -0.13349177584642502
Number of life quality posts: 17
Average sentiment of life-quality-related posts: -0.9116412319541428
Number of aesthetic posts: 7
Average sentiment of aesthetics-related posts: -0.12533501561758817
Number of government posts: 266
Average sentiment of governmental posts: -0.19231961386558713


In [56]:
posts['year'] = pd.to_datetime(posts['date']).dt.year
year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

for theme in theme_cols:
    print(f"Sentiments for {theme}:")
    for date in range(2010, 2027):
        print(f"Amount of posts discussing {theme} in {date}:", len(year_datasets[date].loc[year_datasets[date][theme] == True])) if len(year_datasets[date].loc[year_datasets[date][theme] == True]) > 0 else None
        print(f"Average sentiment for {theme} in {date}:", avg_sentiment_calculation(year_datasets[date], theme)) if len(year_datasets[date].loc[year_datasets[date][theme] == True]) > 0 else None
    print("\n")

Sentiments for environment:
Amount of posts discussing environment in 2025: 8
Average sentiment for environment in 2025: -0.8406611129254491
Amount of posts discussing environment in 2026: 53
Average sentiment for environment in 2026: -0.8058986559846341


Sentiments for infrastructure:
Amount of posts discussing infrastructure in 2022: 1
Average sentiment for infrastructure in 2022: 0.0
Amount of posts discussing infrastructure in 2024: 4
Average sentiment for infrastructure in 2024: -0.19950592991398208
Amount of posts discussing infrastructure in 2025: 65
Average sentiment for infrastructure in 2025: -0.37647420863227554
Amount of posts discussing infrastructure in 2026: 205
Average sentiment for infrastructure in 2026: -0.1806828385775629


Sentiments for housing:
Amount of posts discussing housing in 2024: 1
Average sentiment for housing in 2024: 1.0
Amount of posts discussing housing in 2025: 1
Average sentiment for housing in 2025: -1.0
Amount of posts discussing housing in 2026